# Experiment buffer inspection

Time-series view of **constraint buffer** (CCR) and **shipping buffers** (per product) for a DBR simulation experiment.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from IPython.display import display

from manusim.metrics import ExperimentMetrics

In [ ]:
# --- configure experiment ---
EXPERIMENT_PATH = Path(
    "C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606012136_dbr" #2606012130_dbr" #2606012136_dbr
)
RUN = 1

experiment_path = EXPERIMENT_PATH.resolve()
print(experiment_path)

## Experiment overview

In [ ]:
summary_path = experiment_path / "experiment_summary.yaml"
if summary_path.exists():
    summary = yaml.safe_load(summary_path.read_text())
    display(pd.DataFrame(summary.get("runs_summary", [])))
    print("Average run time (s):", summary["experiment_info"]["average_run_time"])

metadata_path = experiment_path / "experiment_metadata.csv"
if metadata_path.exists():
    display(pd.read_csv(metadata_path))

config = yaml.safe_load((experiment_path / ".hydra" / "config.yaml").read_text())
warmup = config["simulation"]["warmup"]
run_until = config["simulation"]["run_until"]
print(f"Simulation window: warmup={warmup}, run_until={run_until}")
print(
    "Buffer controls:",
    {k: config["simulation"][k] for k in [
        "cb_target_level", "cb_update", "sb_update",
        "buffers_update_multiplyer", "buffers_update_warmup",
    ]},
)

In [ ]:
run_folder = experiment_path / f"run_{RUN:03d}"

dbr_general = pd.read_csv(run_folder / "metrics_dbr_general.csv", index_col=0)
dbr_product = pd.read_csv(run_folder / "metrics_dbr_product.csv", index_col=0)

print("DBR general (run mean):")
display(dbr_general)
print("DBR product (run mean):")
display(dbr_product)

## Load time-series logs

In [ ]:
metrics = ExperimentMetrics(experiment_path)
metrics.read_params()
metrics.read_logs()

logs = metrics.logs.copy()
logs = logs.loc[logs["run"] == RUN].copy()

CONSTRAINT_VARS = [
    "constraintBufferLevel",
    "constraintBufferWip",
    "constraintBufferQueue",
    "constraintBufferTarget",
]
SHIPPING_VARS = ["shippingBufferLevel", "shippingBufferTarget"]
BUFFER_VARS = CONSTRAINT_VARS + SHIPPING_VARS

buffer_logs = logs.loc[logs["variable"].isin(BUFFER_VARS)].copy()
buffer_logs["metric"] = buffer_logs["variable"] + "_" + buffer_logs["key"].str.replace("produto", "p")

products = sorted(
    buffer_logs.loc[buffer_logs["key"] != "general", "key"].unique().tolist()
)
print(f"Buffer log rows: {len(buffer_logs):,}")
buffer_logs.groupby(["variable", "key"]).size()

In [ ]:
def filter_window(df: pd.DataFrame, t_min: float | None = None, t_max: float | None = None) -> pd.DataFrame:
    out = df
    if t_min is not None:
        out = out.loc[out["time"] >= t_min]
    if t_max is not None:
        out = out.loc[out["time"] <= t_max]
    return out


def target_changes(df: pd.DataFrame, variable: str) -> pd.DataFrame:
    """Rows where a buffer target changed (adjustment events)."""
    cols = ["time", "value", "key", "run"]
    tmp = (
        df.loc[df["variable"] == variable, cols]
        .sort_values(["key", "time"])
        .drop_duplicates(subset=["key", "value"], keep="first")
    )
    return tmp


# Analysis window: after warmup (steady-state scheduling region)
ANALYSIS_FROM = warmup
analysis_logs = filter_window(buffer_logs, t_min=ANALYSIS_FROM)
print(f"Analysis from t={ANALYSIS_FROM} ({len(analysis_logs):,} rows)")

## Constraint buffer (CCR)

Level = WIP + queue at the constraint resource. Target is adjusted when `cb_update` is enabled.

In [ ]:
cb = filter_window(
    analysis_logs.loc[
        (analysis_logs["key"] == "general")
        & (analysis_logs["variable"].isin(CONSTRAINT_VARS))
    ],
)

fig = px.line(
    cb,
    x="time",
    y="value",
    color="variable",
    title=f"Constraint buffer — run {RUN} (t ≥ {ANALYSIS_FROM})",
)
fig.update_layout(xaxis_title="simulation time", yaxis_title="hours")
fig.show()

In [ ]:
cb_level = cb.loc[cb["variable"] == "constraintBufferLevel", ["time", "value"]].copy()
cb_target = cb.loc[cb["variable"] == "constraintBufferTarget", ["time", "value"]].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=cb_level["time"], y=cb_level["value"], name="level", mode="lines"))
fig.add_trace(
    go.Scatter(
        x=cb_target["time"],
        y=cb_target["value"],
        name="target",
        mode="lines",
        line=dict(dash="dash"),
    )
)
fig.update_layout(
    title="Constraint buffer level vs target",
    xaxis_title="simulation time",
    yaxis_title="hours",
)
fig.show()

print("Constraint target adjustments:")
display(target_changes(analysis_logs, "constraintBufferTarget"))

In [ ]:
# Utilization relative to target (when target > 0)
level = cb_level.set_index("time")["value"]
target = cb_target.set_index("time")["value"].reindex(level.index, method="ffill")
util = (level / target).replace([float("inf"), -float("inf")], pd.NA)

fig = px.line(
    x=util.index,
    y=util.values,
    title="Constraint buffer utilization (level / target)",
    labels={"x": "simulation time", "y": "level / target"},
)
fig.add_hline(y=1 / 3, line_dash="dot", annotation_text="green zone (1/3)")
fig.add_hline(y=2 / 3, line_dash="dot", annotation_text="red zone (2/3)")
fig.show()

## Shipping buffers (per product)

Level is finished-goods coverage; target moves when penetration rules trigger (`sb_update`).

In [ ]:
sb = filter_window(
    analysis_logs.loc[analysis_logs["variable"].isin(SHIPPING_VARS)],
)

fig = px.line(
    sb,
    x="time",
    y="value",
    color="variable",
    facet_col="key",
    facet_col_wrap=5,
    title=f"Shipping buffer level & target — run {RUN}",
)
fig.update_layout(height=900)
fig.show()

In [ ]:
sb_summary = (
    sb.groupby(["key", "variable"])["value"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(2)
)
display(sb_summary)

print("Shipping target adjustments per product:")
for product in products:
    changes = target_changes(
        analysis_logs.loc[analysis_logs["key"] == product], "shippingBufferTarget"
    )
    if len(changes):
        print(f"\n{product}:")
        display(changes)

In [ ]:
# Initial vs final shipping targets (from time series)
target_series = sb.loc[sb["variable"] == "shippingBufferTarget"].sort_values(["key", "time"])
target_bookends = (
    target_series.groupby("key")["value"]
    .agg(first="first", last="last")
    .assign(delta=lambda d: d["last"] - d["first"])
)
target_bookends.join(
    dbr_product[["shippingBufferTarget"]].rename(columns={"shippingBufferTarget": "run_mean"})
)

## Optional zoom window

Set `ZOOM_FROM` / `ZOOM_TO` to inspect a shorter interval (e.g. around a target change).

In [ ]:
ZOOM_FROM = 100_000
ZOOM_TO = 200_000

zoom = filter_window(analysis_logs, t_min=ZOOM_FROM, t_max=ZOOM_TO)

fig = px.line(
    zoom,
    x="time",
    y="value",
    color="metric",
    title=f"Buffers zoom [{ZOOM_FROM}, {ZOOM_TO}]",
)
fig.show()

## Export plots (optional)

Uncomment to write HTML files under the experiment folder.

In [ ]:
# save_plots = experiment_path / f"plots_buffers_run{RUN:03d}"
# save_plots.mkdir(exist_ok=True, parents=True)
# fig.write_html(save_plots / "constraint_buffer.html")  # re-run fig cell before export